# Evaluate Qwen3 1.7B LoRA Adapters

Evaluate the explicit-theorem and implicit-theorem adapters on the frozen test split.

In [ ]:
!pip install -q -U mlx-lm pandas tqdm

In [ ]:
from pathlib import Path
import sys

from mlx_lm import generate, load
from tqdm.auto import tqdm

In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
from training_eval.eval_utils import (
    default_test_dir,
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_fine_tuned_chat_messages,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B-MLX-bf16"
ADAPTER_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora"
RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_eval"
MAX_NEW_TOKENS = 256

In [ ]:
ADAPTERS = {
    "explicit_theorems": ADAPTER_ROOT / "explicit_theorems" / "adapters",
    "implicit_theorems": ADAPTER_ROOT / "implicit_theorems" / "adapters",
}

In [ ]:
records = load_jsonl_records(default_test_dir(PROJECT_ROOT), pattern="*_preview.jsonl")
len(records)

In [ ]:
def make_qwen_prompt(tokenizer, problem):
    messages = make_fine_tuned_chat_messages(problem)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def load_adapter_model(adapter_path):
    return load(MODEL_NAME, adapter_path=str(adapter_path))

In [ ]:
def generate_answer(model, tokenizer, problem):
    prompt = make_qwen_prompt(tokenizer, problem)
    return generate(model, tokenizer, prompt=prompt, max_tokens=MAX_NEW_TOKENS, verbose=False)

In [ ]:
def evaluate_adapter(adapter_label, adapter_path, records):
    model, tokenizer = load_adapter_model(adapter_path)
    rows = []

    for record in tqdm(records, desc=adapter_label):
        raw_output = generate_answer(model, tokenizer, record["problem"])
        predicted = extract_json_object(raw_output)
        metadata = record.get("metadata", {})

        rows.append({
            "adapter": adapter_label,
            "id": record["id"],
            "family": record["family"],
            "problem_type": record["problem_type"],
            "difficulty": record["difficulty"],
            "answer_type": record["answer_type"],
            "manual_variation": metadata.get("manual_variation", False),
            "problem": record["problem"],
            "canonical_answer": record["canonical_answer"],
            "raw_output": raw_output,
            "predicted_answer": predicted,
            "correct": is_correct(predicted, record["canonical_answer"]),
        })

    return rows

In [ ]:
def save_adapter_eval(adapter_label, rows):
    df = rows_to_frame(rows)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": MODEL_NAME,
        "adapter": adapter_label,
        "dataset": "benchmark/data/test/*_preview.jsonl",
    })
    return save_results(rows, RESULT_ROOT / adapter_label, metrics)

In [ ]:
all_rows = []

for adapter_label, adapter_path in ADAPTERS.items():
    rows = evaluate_adapter(adapter_label, adapter_path, records)
    save_adapter_eval(adapter_label, rows)
    all_rows.extend(rows)

df = rows_to_frame(all_rows)
df.head()

In [ ]:
display(df.groupby("adapter")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby(["adapter", "answer_type"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(df.groupby(["adapter", "family"])["correct"].agg(["mean", "sum", "count"]).sort_index())

In [ ]:
df.loc[~df["correct"], ["adapter", "id", "family", "problem_type", "canonical_answer", "predicted_answer", "raw_output"]].head(30)